# Q学習

Q学習では、更新式が長期報酬の見積もりにどう効くかを、数式とコードの両方で確認します。


## 参考動画
- [対応動画 1](https://www.youtube.com/watch?v=JPt5JYmcngc)
@[youtube](https://www.youtube.com/watch?v=JPt5JYmcngc)

## 参考リンク
- https://www.youtube.com/watch?v=JPt5JYmcngc


## 背景と目的

Q学習は行動価値を直接学び、環境モデルなしで最適行動を探索します。

オフポリシー更新の意味を理解すると、探索戦略の設計自由度が上がります。

行動価値更新と方策抽出を分けて確認します。


## 最初に解きたい疑問

1. なぜ `max` を使うと、オフポリシーになるのか。
2. `Q(s,a)` は、状態の価値 `V(s)` と何が違うのか。
3. `reference_return` を先に計算するのは、何を確認するためなのか。
4. 方策を明示せずに、どうやって最適行動を学ぶのか。
5. 探索率が高くても学習が進むのは、なぜなのか。


## 先に押さえる言葉

- `Q-function`: 状態と行動の組が、将来どれくらい得かを表す関数です。行動を直接比較できます。
- `off-policy`: 実際に行動する方策と、学習で目指す方策が違っていても学べる性質です。Q学習はこれに当たります。
- `TD target`: 更新で目標にする値です。Q学習では報酬に次状態の最大Q値を足した形になります。
- `greedy policy`: 各状態で Q が最大の行動を選ぶ方策です。学習後の行動選択でよく使います。


## 実行前の見取り図

1. `reference_return=` が、割引報酬の基準値として出ているか。
2. `Q(s0,right)=` の更新後値が、TD target に合わせて変わっているか。
3. `choose_action(...)` の結果で、探索と活用が切り替わっているか。


## つまずきやすい点

- `max_{a'}` が『実際に選んだ行動』ではない点を、on-policy との対比で説明する補足がない。
- 最後に方策をどう抽出するかを、Q値の一覧から結びつける説明が足りない。


## 検証 1: Q学習の更新準備

Q学習の更新式確認のため、割引率と報酬列を先に定義します。


In [ ]:
rewards = [0.0, 1.0, 0.2, 1.4]
gamma = 0.91
g = 0.0
for r in reversed(rewards):
    g = r + gamma * g
print('task = q-learning', 'reference_return=', round(g, 6))

次セルでmax演算を含む更新規則に接続します。



## 検証 2: ベルマン更新を1回行う

次に、価値更新を1ステップだけ計算します。1回更新でも、再帰構造の意味は十分に見えてきます。


In [ ]:
v_next = {'s0': 0.4, 's1': 0.8}
reward = {'left': 0.2, 'right': 1.0}
trans = {'left': 's0', 'right': 's1'}
v_s = max(reward[a] + gamma * v_next[trans[a]] for a in ['left', 'right'])
print('updated V(s)=', round(v_s, 6))

ベルマン更新は『今の価値』を『次状態の価値』で再定義する操作です。この再帰が強化学習の中心です。



## 定義の確認

1. $Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma\max_{a'}Q(s',a') - Q(s,a)]$


## 検証 3: Q値更新を比較する

ここで Q学習の更新式をコードに写し、数値の動きを確認します。式を読むだけでは掴みにくい感覚を得る段階です。


In [ ]:
Q = {('s0','left'): 0.3, ('s0','right'): 0.1, ('s1','left'): 0.5, ('s1','right'): 0.7}
alpha = 0.2
r, s, a, s_next = 1.0, 's0', 'right', 's1'
td_target = r + gamma * max(Q[(s_next,'left')], Q[(s_next,'right')])
Q[(s,a)] += alpha * (td_target - Q[(s,a)])
print('Q(s0,right)=', round(Q[(s,a)], 6))

更新後の値が過去の値とどれだけ違うかは、学習率と TD 誤差で決まります。ここが調整ポイントです。



## 検証 4: 探索と活用の切り替え

次に、探索率を変えたときの行動選択を見ます。探索不足は局所最適に閉じる典型的な原因です。


In [ ]:
import random

random.seed(7)

def choose_action(q_left, q_right, epsilon):
    if random.random() < epsilon:
        return random.choice(['left', 'right'])
    return 'left' if q_left >= q_right else 'right'

samples_high_eps = [choose_action(0.4, 0.7, 0.5) for _ in range(8)]
samples_low_eps = [choose_action(0.4, 0.7, 0.1) for _ in range(8)]
print('epsilon=0.5 ->', samples_high_eps)
print('epsilon=0.1 ->', samples_low_eps)


探索率は固定せず、学習段階に応じて減衰させるのが一般的です。初期は広く探索し、後半で活用へ寄せます。



## 検証 5: 方策評価の簡易チェック

最後に、方策の平均報酬を簡易的に比較します。アルゴリズムの評価は、更新式だけでなく結果の検証が不可欠です。


In [ ]:
episode_rewards = [1.2, 0.8, 1.5, 1.1, 1.4]
avg_reward = sum(episode_rewards) / len(episode_rewards)
variance = sum((r - avg_reward) ** 2 for r in episode_rewards) / len(episode_rewards)
print('avg =', round(avg_reward, 4))
print('var =', round(variance, 4))

平均だけでなく分散を見ると、方策の安定性も評価できます。実運用ではこの二軸が重要です。



## まとめ

今回のノートで押さえておくべき誤解しやすい点を整理します。

1. 探索率が低すぎて行動が固定化する
2. 報酬設計が目的とずれている
3. 長期ロールアウトの不安定性を検証しない
